# 05 — Independent validation, test evaluation, codebook diagnostics, and promotion gate

This notebook evaluates the exact checkpoint on separate provenance-checked validation and test manifests. Reports include per-task loss/reconstruction, modality coverage, and held-out vision/audio/motion codebook usage, normalized entropy, perplexity, dead-code count, and maximum assignment concentration. Both splits must use the same checkpoint hash but independent data fingerprints. Promotion is blocked until verified data and reviewed numeric thresholds for loss and codebook collapse are approved; a successful workflow or lower loss is not sufficient.


In [ ]:
import sys, json
from pathlib import Path

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "configs" / "phone-research.yaml").is_file() and (candidate / "src" / "onecore").is_dir():
            return candidate
    raise RuntimeError(f"Could not locate OneCore project root from {current}")

root = find_project_root()
print("OneCore project root:", root)
sys.path.insert(0, str(root / "src"))

from onecore.checkpoint import write_json_atomic
from onecore.evaluate import evaluate_checkpoint, promotion_gate
from onecore.promote import promote_candidate

config = root / "configs" / "phone-research.yaml"
checkpoint = root / "run-state" / "checkpoint.pt"
tokenizer = root / "run-state" / "tokenizer.json"
thresholds = root / "configs" / "promotion-thresholds.json"

validation_metrics = evaluate_checkpoint(
    root, config, checkpoint, tokenizer, thresholds,
    root / "run-state" / "evaluation.validation.json", split="validation",
)
test_metrics = evaluate_checkpoint(
    root, config, checkpoint, tokenizer, thresholds,
    root / "run-state" / "evaluation.test.json", split="test",
)

metrics = dict(validation_metrics)
metrics["test_metrics"] = test_metrics
metrics["test_manifest_sha256"] = test_metrics.get("split_manifest_sha256")
metrics["test_split_independent"] = (
    validation_metrics.get("split_manifest_sha256") != test_metrics.get("split_manifest_sha256")
)
metrics["evaluation_bundle_id"] = (
    f"{validation_metrics['evaluation_id']}+{test_metrics['evaluation_id']}"
)
write_json_atomic(root / "run-state" / "evaluation.json", metrics)

passed, failures = promotion_gate(metrics)
print(json.dumps({"metrics": metrics, "promotion_gate_passed": passed, "failures": failures}, ensure_ascii=False, indent=2))
if passed:
    print(promote_candidate(root / "run-state" / "evaluation.json", root / "run-state" / "model-registry.json"))
else:
    print("PROMOTION BLOCKED. No candidate was merged or marked as released.")

